# 🎨 Feature Engineering — Best Artworks of All Time

This notebook performs **Feature Engineering** on the Best Artworks dataset.

We extract features from 3 sources:
1. 📁 **Filename** → artist name, artwork ID
2. 📋 **artists.csv** → genre, nationality, birth/death year, art movement era
3. 🖼️ **Image pixels** → brightness, contrast, saturation, RGB stats, aspect ratio

Then we **engineer new features** and **encode** everything for ML use.

## 📦 Step 1 — Install & Import Libraries

In [1]:
# 1. Create conda environment with Python 3.9 (compatible with StyleGAN2-ADA)
!conda create -n stylegan_env python=3.9 -y
!conda activate stylegan_env

# 2. Install PyTorch with CUDA 11.8 (stable with StyleGAN2-ADA)
!pip install torch==2.0.1+cu118 torchvision==0.15.2+cu118 --index-url https://download.pytorch.org/whl/cu118

# 3. Install dependencies
!pip install click requests tqdm pyspng imageio-ffmpeg numpy==1.23.5 ninja

# 4. Clone StyleGAN2-ADA
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git
%cd stylegan2-ada-pytorch

SyntaxError: invalid syntax (4126057230.py, line 2)

In [ ]:
!pip install pandas numpy matplotlib seaborn pillow scikit-learn kagglehub -q

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

print("✅ All libraries imported successfully!")

## 📥 Step 2 — Download Dataset

In [ ]:
import kagglehub

path = kagglehub.dataset_download("ikarus777/best-artworks-of-all-time")
print("📂 Dataset path:", path)

# Paths
csv_path      = os.path.join(path, "artists.csv")
resized_path  = os.path.join(path, "resized", "resized")

print("📄 CSV:", csv_path)
print("🖼️  Images folder:", resized_path)

## 📋 Step 3 — Load artists.csv & Explore

In [ ]:
artists_meta = pd.read_csv(csv_path)
print("Shape:", artists_meta.shape)
print("Columns:", artists_meta.columns.tolist())
artists_meta.head()

In [ ]:
# Check missing values
print("Missing values per column:")
print(artists_meta.isnull().sum())

## 📁 Step 4 — Build Image-Level DataFrame from Filenames

كل صورة اسمها بيكون: `ArtistName_ID.jpg`  
مثلاً: `Vincent_van_Gogh_511.jpg`

هنستخرج منه: اسم الفنان + رقم اللوحة.

In [ ]:
image_files = [f for f in os.listdir(resized_path) if f.endswith(".jpg")]
print(f"Total images found: {len(image_files)}")

records = []
for fname in image_files:
    # Extract parts: everything before the last _ is the artist name
    parts = fname.replace(".jpg", "").rsplit("_", 1)
    if len(parts) == 2 and parts[1].isdigit():
        artist_raw = parts[0]          # e.g. 'Vincent_van_Gogh'
        artwork_id = int(parts[1])     # e.g. 511
    else:
        artist_raw = fname.replace(".jpg", "")
        artwork_id = -1

    records.append({
        "filename"   : fname,
        "filepath"   : os.path.join(resized_path, fname),
        "artist_raw" : artist_raw,
        "artwork_id" : artwork_id,
    })

df = pd.DataFrame(records)
print(df.shape)
df.head()

## 🔤 Step 5 — Feature Engineering from Filename

نستخرج: First Name, Last Name, وعدد الكلمات في الاسم.

In [ ]:
# Split artist name into first and last name
def parse_artist_name(raw_name):
    parts = raw_name.split("_")
    first_name = parts[0] if len(parts) >= 1 else ""
    last_name  = parts[-1] if len(parts) >= 2 else ""
    full_name  = " ".join(parts)
    return first_name, last_name, full_name

df[["first_name", "last_name", "artist_name"]] = df["artist_raw"].apply(
    lambda x: pd.Series(parse_artist_name(x))
)

# Name length as a simple feature
df["name_word_count"] = df["artist_raw"].apply(lambda x: len(x.split("_")))

print("Sample features from filename:")
df[["filename", "artist_name", "first_name", "last_name", "artwork_id", "name_word_count"]].head(8)

## 📋 Step 6 — Merge with artists.csv Metadata

الـ CSV بيحتوي على: `name`, `years`, `genre`, `nationality`, `bio`, `wikipedia`, `paintings`  
هنعمل merge عشان نضيف هذه المعلومات لكل صورة.

In [ ]:
# Normalize artist name in CSV to match filename format
artists_meta["artist_raw"] = artists_meta["name"].str.replace(" ", "_")

# Keep only relevant columns
meta_cols = ["artist_raw", "years", "genre", "nationality", "paintings"]
artists_slim = artists_meta[meta_cols].copy()

# Merge on artist_raw
df = df.merge(artists_slim, on="artist_raw", how="left")

print("Shape after merge:", df.shape)
print("Columns now:", df.columns.tolist())
df.head(3)

## 📅 Step 7 — Feature Engineering from `years` column

عمود `years` بيكون فيه حاجة زي `"1853 - 1890"` — هنستخرج منه:  
`birth_year`, `death_year`, `age_at_death`, `is_alive`

In [ ]:
def parse_years(years_str):
    """Extract birth and death year from string like '1853 - 1890'"""
    if pd.isna(years_str):
        return pd.Series([None, None, None, 0])

    numbers = re.findall(r"\d{4}", str(years_str))

    birth_year  = int(numbers[0]) if len(numbers) >= 1 else None
    death_year  = int(numbers[1]) if len(numbers) >= 2 else None
    age_at_death = (death_year - birth_year) if (birth_year and death_year) else None
    is_alive    = 1 if death_year is None else 0

    return pd.Series([birth_year, death_year, age_at_death, is_alive])

df[["birth_year", "death_year", "age_at_death", "is_alive"]] = df["years"].apply(parse_years)

print("Sample year features:")
df[["artist_name", "years", "birth_year", "death_year", "age_at_death", "is_alive"]].drop_duplicates("artist_name").head(8)

## 🏛️ Step 8 — Art Movement Era Feature

بالعربي: هنحدد الفنان كان بيشتغل في أنهي عصر فني؟  
بنستخدم `birth_year` عشان نصنفه:

| Era | Years |
|-----|-------|
| Medieval | قبل 1400 |
| Renaissance | 1400-1600 |
| Baroque | 1600-1750 |
| Romanticism | 1750-1850 |
| Impressionism | 1850-1900 |
| Modern | 1900+ |

In [ ]:
def get_art_era(birth_year):
    if pd.isna(birth_year):
        return "Unknown"
    y = int(birth_year)
    if y < 1400:
        return "Medieval"
    elif y < 1600:
        return "Renaissance"
    elif y < 1750:
        return "Baroque"
    elif y < 1850:
        return "Romanticism"
    elif y < 1900:
        return "Impressionism"
    else:
        return "Modern"

df["art_era"] = df["birth_year"].apply(get_art_era)

print("Art era distribution:")
print(df["art_era"].value_counts())

## 🖼️ Step 9 — Image-Level Feature Extraction

هنفتح كل صورة ونستخرج منها:

| Feature | المعنى |
|---------|--------|
| `img_width`, `img_height` | أبعاد الصورة |
| `aspect_ratio` | width / height |
| `mean_r`, `mean_g`, `mean_b` | متوسط كل قناة لون |
| `std_r`, `std_g`, `std_b` | التباين في كل قناة |
| `brightness` | متوسط الصورة بعد تحويلها لـ grayscale |
| `contrast` | std الصورة الـ grayscale |
| `saturation` | متوسط قناة S في HSV |
| `colorfulness` | درجة غنى الألوان |

> ⚠️ هنشتغل على sample صغيرة (500 صورة) بدل الـ 8683 عشان يكون سريع في التشغيل. تقدر تشيل `[:500]` عشان تعمل الكل.

In [ ]:
def extract_image_features(filepath):
    """Extract pixel-level features from a single image."""
    try:
        img_rgb = Image.open(filepath).convert("RGB")
        img_arr = np.array(img_rgb, dtype=np.float32)

        # --- Dimensions ---
        h, w, _ = img_arr.shape
        aspect_ratio = round(w / h, 4)

        # --- RGB channel stats ---
        r, g, b = img_arr[:,:,0], img_arr[:,:,1], img_arr[:,:,2]
        mean_r, mean_g, mean_b = r.mean(), g.mean(), b.mean()
        std_r,  std_g,  std_b  = r.std(),  g.std(),  b.std()

        # --- Brightness (grayscale mean) ---
        gray = 0.299*r + 0.587*g + 0.114*b
        brightness = gray.mean()
        contrast   = gray.std()

        # --- Saturation via HSV ---
        img_hsv  = img_rgb.convert("HSV") if hasattr(img_rgb, "convert") else img_rgb
        img_hsv  = np.array(Image.open(filepath).convert("HSV"), dtype=np.float32)
        saturation = img_hsv[:,:,1].mean()
        hue_mean   = img_hsv[:,:,0].mean()

        # --- Colorfulness (Hasler & Suesstrunk formula) ---
        rg = r - g
        yb = 0.5 * (r + g) - b
        colorfulness = np.sqrt(rg.std()**2 + yb.std()**2) + 0.3 * np.sqrt(rg.mean()**2 + yb.mean()**2)

        return {
            "img_width"   : w,
            "img_height"  : h,
            "aspect_ratio": aspect_ratio,
            "mean_r"      : round(mean_r, 3),
            "mean_g"      : round(mean_g, 3),
            "mean_b"      : round(mean_b, 3),
            "std_r"       : round(std_r, 3),
            "std_g"       : round(std_g, 3),
            "std_b"       : round(std_b, 3),
            "brightness"  : round(brightness, 3),
            "contrast"    : round(contrast, 3),
            "saturation"  : round(saturation, 3),
            "hue_mean"    : round(hue_mean, 3),
            "colorfulness": round(colorfulness, 3),
        }
    except Exception as e:
        return None  # Skip corrupted images


# ⚠️ Run on a sample for speed — remove [:500] to process all images
print("🔄 Extracting image features (sample of 500)...")
sample_df = df.sample(500, random_state=42).copy()

feature_list = []
for idx, row in sample_df.iterrows():
    feats = extract_image_features(row["filepath"])
    if feats:
        feats["filename"] = row["filename"]
        feature_list.append(feats)

img_features_df = pd.DataFrame(feature_list)
print(f"✅ Extracted features for {len(img_features_df)} images")
img_features_df.head()

## 🔗 Step 10 — Merge Image Features into Main DataFrame

In [ ]:
# Merge pixel features back into main df
df_full = sample_df.merge(img_features_df, on="filename", how="inner")
print("Shape after merging image features:", df_full.shape)
df_full.head(3)

## 🎭 Step 11 — Derived / Engineered Features

دلوقتي هنبني features جديدة من اللي استخرجناه.

In [ ]:
# --- 1. Emotion Label (brightness-based heuristic) ---
# فكرة: الصورة الفاتحة = فرح، المتوسطة = هدوء، الداكنة = حزن
def get_emotion(brightness):
    if brightness >= 170:
        return "Joy"
    elif brightness >= 100:
        return "Calm"
    else:
        return "Sadness"

df_full["emotion_label"] = df_full["brightness"].apply(get_emotion)

# --- 2. Brightness Category ---
df_full["brightness_cat"] = pd.cut(
    df_full["brightness"],
    bins=[0, 85, 170, 255],
    labels=["Dark", "Medium", "Bright"]
)

# --- 3. Color Temperature (Warm vs Cool) ---
# Warm = Red/Yellow dominant (R > B), Cool = Blue dominant (B > R)
df_full["color_temp"] = df_full.apply(
    lambda row: "Warm" if row["mean_r"] > row["mean_b"] else "Cool", axis=1
)

# --- 4. Is Portrait (Height > Width) ---
df_full["is_portrait"] = (df_full["img_height"] > df_full["img_width"]).astype(int)

# --- 5. Is Colorful (colorfulness > threshold) ---
color_thresh = df_full["colorfulness"].median()
df_full["is_colorful"] = (df_full["colorfulness"] > color_thresh).astype(int)

# --- 6. Dominant Channel ---
def dominant_channel(r, g, b):
    if r >= g and r >= b:
        return "Red"
    elif g >= r and g >= b:
        return "Green"
    else:
        return "Blue"

df_full["dominant_channel"] = df_full.apply(
    lambda row: dominant_channel(row["mean_r"], row["mean_g"], row["mean_b"]), axis=1
)

# --- 7. Productivity Score = paintings / active_years ---
df_full["active_years"] = df_full["death_year"] - df_full["birth_year"]
df_full["productivity"] = df_full.apply(
    lambda row: round(row["paintings"] / row["active_years"], 3)
    if pd.notna(row["active_years"]) and row["active_years"] > 0 else None,
    axis=1
)

print("✅ Derived features added!")
print(df_full[["filename", "emotion_label", "brightness_cat", "color_temp", 
               "is_portrait", "is_colorful", "dominant_channel", "productivity"]].head(8))

## 🔢 Step 12 — Encoding Categorical Features

الموديلات مش بتفهم نص — كل حاجة لازم تتحول لأرقام.

| Encoding | متى نستخدمه |
|----------|-------------|
| **Label Encoding** | لما عدد الفئات كبير (artist, nationality) |
| **One-Hot Encoding** | لما عدد الفئات صغير (emotion, color_temp, brightness_cat) |

In [ ]:
le = LabelEncoder()

# --- Label Encoding ---
label_encode_cols = ["artist_name", "nationality", "art_era", "dominant_channel"]

for col in label_encode_cols:
    if col in df_full.columns:
        df_full[f"{col}_enc"] = le.fit_transform(df_full[col].astype(str))

print("✅ Label Encoding done for:", label_encode_cols)

# --- One-Hot Encoding ---
onehot_cols = ["emotion_label", "color_temp", "brightness_cat"]
df_full = pd.get_dummies(df_full, columns=onehot_cols, drop_first=False)

print("✅ One-Hot Encoding done for:", onehot_cols)
print("\nNew shape:", df_full.shape)

## 📏 Step 13 — Normalize Numerical Features

الـ MinMaxScaler بيحول كل الأرقام للنطاق [0, 1] عشان مفيش feature يهيمن على التانية.

In [ ]:
num_cols = ["brightness", "contrast", "saturation", "colorfulness",
            "mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b",
            "aspect_ratio", "hue_mean"]

# Only normalize columns that exist
num_cols_present = [c for c in num_cols if c in df_full.columns]

scaler = MinMaxScaler()
df_full[[f"{c}_norm" for c in num_cols_present]] = scaler.fit_transform(df_full[num_cols_present])

print("✅ Normalized features:")
df_full[[f"{c}_norm" for c in num_cols_present]].describe().round(3)

## 📊 Step 14 — Visualize Engineered Features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Feature Engineering — Visual Overview", fontsize=16, fontweight="bold")

# 1. Emotion Label Distribution
emotion_cols = [c for c in df_full.columns if c.startswith("emotion_label_")]
if emotion_cols:
    emotion_counts = df_full[emotion_cols].sum()
    emotion_counts.index = [c.replace("emotion_label_", "") for c in emotion_counts.index]
    emotion_counts.plot(kind="bar", ax=axes[0,0], color=["gold", "steelblue", "mediumpurple"])
    axes[0,0].set_title("Emotion Label Distribution")
    axes[0,0].set_xlabel("Emotion")
    axes[0,0].tick_params(axis="x", rotation=0)

# 2. Brightness Distribution
axes[0,1].hist(df_full["brightness"], bins=30, color="coral", edgecolor="black")
axes[0,1].set_title("Brightness Distribution")
axes[0,1].set_xlabel("Brightness (0-255)")
axes[0,1].axvline(df_full["brightness"].mean(), color="black", linestyle="--", label="Mean")
axes[0,1].legend()

# 3. Colorfulness Distribution
axes[0,2].hist(df_full["colorfulness"], bins=30, color="mediumseagreen", edgecolor="black")
axes[0,2].set_title("Colorfulness Distribution")
axes[0,2].set_xlabel("Colorfulness Score")

# 4. Color Temperature
color_temp_cols = [c for c in df_full.columns if c.startswith("color_temp_")]
if color_temp_cols:
    ct_counts = df_full[color_temp_cols].sum()
    ct_counts.index = [c.replace("color_temp_", "") for c in ct_counts.index]
    ct_counts.plot(kind="pie", ax=axes[1,0], autopct="%1.1f%%",
                   colors=["tomato", "cornflowerblue"])
    axes[1,0].set_title("Color Temperature (Warm vs Cool)")
    axes[1,0].set_ylabel("")

# 5. Art Era Distribution
era_counts = df_full["art_era"].value_counts()
era_counts.plot(kind="barh", ax=axes[1,1], color="mediumpurple")
axes[1,1].set_title("Art Era Distribution")
axes[1,1].set_xlabel("Count")

# 6. Brightness vs Colorfulness scatter
axes[1,2].scatter(df_full["brightness"], df_full["colorfulness"],
                  alpha=0.4, color="darkorange", s=10)
axes[1,2].set_title("Brightness vs Colorfulness")
axes[1,2].set_xlabel("Brightness")
axes[1,2].set_ylabel("Colorfulness")

plt.tight_layout()
plt.show()

## 📊 Step 15 — Feature Correlation Heatmap

In [ ]:
# Select key numerical features for correlation
corr_features = ["brightness", "contrast", "saturation", "colorfulness",
                 "mean_r", "mean_g", "mean_b", "aspect_ratio",
                 "is_portrait", "is_colorful"]

corr_features_present = [c for c in corr_features if c in df_full.columns]
corr_matrix = df_full[corr_features_present].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, linewidths=0.5)
plt.title("Feature Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 💾 Step 16 — Save Final Feature DataFrame

In [ ]:
# Select final feature columns to save
# (drop raw/intermediate columns we don't need for ML)
drop_cols = ["filepath", "artist_raw", "years", "bio"] if "bio" in df_full.columns else ["filepath", "artist_raw", "years"]
drop_cols = [c for c in drop_cols if c in df_full.columns]

df_final = df_full.drop(columns=drop_cols)

# Save to CSV
output_path = "/content/artworks_features.csv"
df_final.to_csv(output_path, index=False)

print(f"✅ Saved! Shape: {df_final.shape}")
print(f"📄 File: {output_path}")
print("\n📋 Final columns:")
print(df_final.columns.tolist())

## 📋 Summary — All Engineered Features

| Feature | Source | Type | Notes |
|---------|--------|------|-------|
| `artist_name` | Filename | Categorical | اسم الفنان كامل |
| `first_name` | Filename | Categorical | الاسم الأول |
| `last_name` | Filename | Categorical | اسم العيلة |
| `artwork_id` | Filename | Numerical | رقم اللوحة |
| `genre` | artists.csv | Categorical | نوع الفن (Impressionism, etc.) |
| `nationality` | artists.csv | Categorical | جنسية الفنان |
| `birth_year` | artists.csv | Numerical | سنة الميلاد |
| `death_year` | artists.csv | Numerical | سنة الوفاة |
| `age_at_death` | Derived | Numerical | عمر الفنان لما مات |
| `is_alive` | Derived | Binary | 1 لو لسه حي |
| `active_years` | Derived | Numerical | عدد سنوات النشاط |
| `art_era` | Derived | Categorical | العصر الفني |
| `productivity` | Derived | Numerical | لوحات / سنة |
| `img_width` | Pixel | Numerical | عرض الصورة |
| `img_height` | Pixel | Numerical | طول الصورة |
| `aspect_ratio` | Pixel | Numerical | نسبة الأبعاد |
| `mean_r/g/b` | Pixel | Numerical | متوسط كل قناة لون |
| `std_r/g/b` | Pixel | Numerical | انتشار كل قناة لون |
| `brightness` | Pixel | Numerical | متوسط الإضاءة |
| `contrast` | Pixel | Numerical | درجة التباين |
| `saturation` | Pixel | Numerical | غنى الألوان (HSV) |
| `hue_mean` | Pixel | Numerical | متوسط درجة اللون (HSV) |
| `colorfulness` | Pixel | Numerical | درجة التلوين الكلية |
| `emotion_label` | Derived | Categorical → OHE | Joy / Calm / Sadness |
| `brightness_cat` | Derived | Categorical → OHE | Dark / Medium / Bright |
| `color_temp` | Derived | Categorical → OHE | Warm / Cool |
| `is_portrait` | Derived | Binary | 1 لو الصورة portrait |
| `is_colorful` | Derived | Binary | 1 لو فوق المتوسط |
| `dominant_channel` | Derived | Categorical → Enc | R / G / B |
| `*_enc` | Label Encoded | Numerical | كل الـ categorical |
| `*_norm` | Normalized | Numerical [0-1] | كل الـ numerical |